# Lesson 03 - Агентні шаблони проектування

У цьому уроці ми розглянемо три основні шаблони проектування для створення ефективних AI-агентів:

1. **Чіткі інструкції для агента** — створення точних, що визначають роль, підказок, які керують поведінкою агента
2. **Структурований вивід з моделями Pydantic** — забезпечення того, що агенти повертають передбачувані, перевірені дані
3. **Агенти з одною відповідальністю** — проєктування цілеспрямованих агентів, які кожен добре виконують одну задачу

Ми застосуємо кожен шаблон до сценарію **рекомендації туристичних напрямків**, поступово створюючи систему, яка може пропонувати напрямки, перевіряти доступність і обробляти логістику.


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Шаблон 1: Чіткі інструкції для агента

Найбільш ефективним шаблоном є також найпростіший: написання чітких, детальних інструкцій для вашого агента.

Добрі інструкції визначають:
- **Хто** є агентом (персона та тон)
- **Що** він має робити (покрокові обов’язки)
- **Як** він має себе поводити (обмеження та стиль)

Нижче ми створюємо агента туристичного консьєржа з явними інструкціями, які формують кожну відповідь, яку він генерує.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Структурований вивід з моделями Pydantic

Вільний текст корисний для розмови, але наступні системи потребують структурованих даних.  
Поєднуючи **моделі Pydantic** з **функцією інструменту**, ми можемо:

- Визначити точну схему для виводу агента  
- Автоматично перевіряти відповіді  
- Надійно інтегрувати результати агента у логіку застосунку  

Ми також представляємо інструмент, який повертає деталі місця призначення, щоб агент базував свої рекомендації на реальних даних.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Агенти з одною відповідальністю

Складні завдання виграють від розподілу роботи між кількома сфокусованими агентами, кожен з яких має одну відповідальність:

- **Експерт з місця призначення**, який знає про місця та їх доступність
- **Планувальник логістики**, який займається рейсами, готелями та маршрутами

Це відображає принцип інженерії програмного забезпечення *розподілу відповідальностей* — кожен агент легше тестується, підтримується та покращується незалежно.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Підсумок

У цьому уроці ми застосували три агентні патерни дизайну до сценарію рекомендера подорожей:

| Патерн | Ключова ідея | Користь |
|---|---|---|
| **Чіткі інструкції** | Визначити персонажа, обов’язки та обмеження заздалегідь | Послідовна, відповідна бренду поведінка агента |
| **Структурований вихід** | Використовувати моделі Pydantic як формат відповіді | Перевірені, машинозчитувані результати |
| **Одна відповідальність** | Дати кожному агенту одну чітку задачу | Легше тестувати, підтримувати та компонувати |

Ці патерни природно поєднуються — ви можете комбінувати чіткі інструкції зі структурованим виходом у агенті з одною відповідальністю, щоб створювати надійні, готові до виробництва системи.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, просимо враховувати, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується звертатися до професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння або неправильне тлумачення, що виникли через використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
